In [7]:
# 1. تثبيت النسخ الجاهزة المتوافقة مع كولاب مباشرة
!pip install --no-deps bitsandbytes accelerate peft trl
!pip install --no-deps unsloth
!pip install xformers --index-url https://download.pytorch.org/whl/cu121
!pip install datasets huggingface_hub

Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement xformers (from versions: none)
ERROR: No matching distribution found for xformers


In [8]:
import torch

# التأكد من عمل كارت الشاشة GPU المخصص (T4 أو A100)
if torch.cuda.is_available():
    print(f"✅ GPU متوفر: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ تحذير: GPU غير متوفر! تأكد من تغيير Runtime Type إلى T4 GPU على Colab.")

✅ GPU متوفر: Tesla T4


In [9]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 1. تحميل النموذج المكمم الأساسي
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 2. إعداد مصفوفات LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ تم إعداد النموذج والطبقات بنجاح.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ تم إعداد النموذج والطبقات بنجاح.


In [10]:
from datasets import load_dataset

print("⏳ جاري تحميل وتنسيق داتا Text-to-SQL المجهزة بالـ CREATE TABLE Schema...")

# تحميل الداتاسيت القياسية الأكثر استقراراً ودقة
dataset = load_dataset("b-mc2/sql-create-context", split="train")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a text-to-SQL system. Given the database schema context, convert the natural language question into a valid SQLite SQL query.

Database Schema:
{schema}

### Input:
{question}

### Response:
{query}"""

EOS_TOKEN = "<|endoftext|>"

def format_prompts(examples):
    schemas = examples["context"]      # كود الـ DDL CREATE TABLE للجداول
    questions = examples["question"]   # السؤال باللغة الطبيعية
    queries = examples["answer"]       # كود الـ SQL الصحيح

    texts = []
    for schema, question, query in zip(schemas, questions, queries):
        text = alpaca_prompt.format(
            schema=schema if schema else "",
            question=question,
            query=query
        ) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

print(f"✅ تم تجهيز وتنسيق البيانات بنجاح! إجمالي العينات: {len(formatted_dataset)}")
print("\n--- 📄 عينة من البيانات المنسقة ---")
print(formatted_dataset[0]["text"][:600] + "\n...")

⏳ جاري تحميل وتنسيق داتا Text-to-SQL المجهزة بالـ CREATE TABLE Schema...
✅ تم تجهيز وتنسيق البيانات بنجاح! إجمالي العينات: 78577

--- 📄 عينة من البيانات المنسقة ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a text-to-SQL system. Given the database schema context, convert the natural language question into a valid SQLite SQL query.

Database Schema:
CREATE TABLE head (age INTEGER)

### Input:
How many heads of the departments are older than 56 ?

### Response:
SELECT COUNT(*) FROM head WHERE age > 56<|endoftext|>
...


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,  # عدد الخطوات التجريبية
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("🚀 بدء عملية التدريب...")
trainer_stats = trainer.train()
print("🎉 اكتمل التدريب بنجاح!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/78577 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 بدء عملية التدريب...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 78,577 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.478311
10,1.846832
15,1.027392
20,0.746810
25,0.623151
30,0.603862
35,0.578945
40,0.569321
45,0.586417
50,0.548614


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


🎉 اكتمل التدريب بنجاح!


In [13]:
from huggingface_hub import notebook_login

# 1. تسجيل الدخول إلى Hugging Face
notebook_login()

# 2. تحديد اسم الريبو على حسابك في Hugging Face
HF_REPO_NAME = "Abdallah-Nabil/Qwen2.5-Coder-7B-Text2SQL-LoRA"

# حفظ ورفع الـ LoRA Adapters فقط (حجمها خفيف حوالي 100-200 ميجابايت)
model.push_to_hub_merged(HF_REPO_NAME, tokenizer, save_method="lora")
print(f"✅ تم رفع الأوزان بنجاح إلى: https://huggingface.co/{HF_REPO_NAME}")

Unsloth: Restored added_tokens_decoder metadata in Abdallah-Nabil/Qwen2.5-Coder-7B-Text2SQL-LoRA/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   1%|          | 7.42MB / 1.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [11:18<00:00, 169.60s/it]


Unsloth: Merge process complete. Saved to `/content/Abdallah-Nabil/Qwen2.5-Coder-7B-Text2SQL-LoRA`
✅ تم رفع الأوزان بنجاح إلى: https://huggingface.co/Abdallah-Nabil/Qwen2.5-Coder-7B-Text2SQL-LoRA
